## 1) Load API keys

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads your .env file automatically

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

## 2) Define LLM's prompt

In [2]:
SYSTEM_PROMPT = """Answer the given question. You must conduct reasoning inside <think> and </think> first every time you get new information. After reasoning, if you find you lack some knowledge, you can call a search engine by <search> query </search>, and it will return the top searched results between <information> and </information>. You can search as many times as you want. If you find no further external knowledge needed, you can directly provide the answer inside <answer> and </answer> without detailed illustrations. For example, <answer> xxx </answer>."""

## 3) Setup the Search Tool (Tavily API)

In [3]:
# pip install openai tavily-python sentence-transformers datasets torch

In [4]:
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def search(query: str, max_results: int = 3) -> str:
    results = tavily.search(query=query, max_results=max_results)
    passages = [r["content"] for r in results["results"]]
    return "\n".join(passages)

## Testing out the search feature

In [5]:
result = search("Who is the head coach of the New England Patriots?")
print(result)

The 16th and current head coach is Mike Vrabel, who was hired on January 12, 2025.
The New England Patriots announce Mike Vrabel as the 16th head coach in team history. Take a look back at Vrabel's iconic career in New England.
# New England Patriots head coach Mike Vrabel at Super Bowl 2026. * Patriots head coach Mike Vrabel was once a linebacker for the team. In his first year at the helm of the New England Patriots, Mike Vrabel is now preparing the team to take on the Seattle Seahawks in Super Bowl 60, with second-year quarterback Drake Maye leading the way for the offense. ## When did Vrabel play for the Patriots? Vrabel played the linebacker position on the team from 2001-2008, earning three Super Bowl rings with the team in 2001, 2003 and 2004. ## Where was Vrabel before he joined the Patriots? ## What were Vrabel's statistics as a linebacker with the Patriots? ## Did Vrabel play anywhere after he left the Patriots? ## When did Vrabel first become a coach? After meeting at Ohio S

In [6]:
results = tavily.search(query="capital of France", max_results=3)

print(f"Number of results: {len(results['results'])}")
for i, r in enumerate(results["results"]):
    print(f"\n--- Passage {i+1} ---")
    print(f"Title:   {r['title']}")
    print(f"URL:     {r['url']}")
    print(f"Content: {r['content'][:200]}...")  # first 200 chars

Number of results: 3

--- Passage 1 ---
Title:   List of capitals of France
URL:     https://en.wikipedia.org/wiki/List_of_capitals_of_France
Content: * Paris (987–1419), the residence of the Kings of France, although they were consecrated at Reims. * Troyes (1419–1422), for a short time during the Hundred Years' War, the city was the seat of the ro...

--- Passage 2 ---
Title:   Paris facts: the capital of France in history
URL:     https://home.adelphi.edu/~ca19535/page%204.html
Content: ## Paris facts: Paris, the **capital of France**. Paris is the **capital of France**, the largest country of Europe with 550 000 km2 (65 millions inhabitants). Paris has 2.234 million inhabitants end ...

--- Passage 3 ---
Title:   Paris
URL:     https://en.wikipedia.org/wiki/Paris
Content: Paris is the capital and largest city of France, with an estimated city population of 2,047,602 in an area of 105.4 km2 (40.7 sq mi), and a metropolitan...


In [7]:
query = "What year was the director of Singin in the Rain born?"
result = search(query)
print(f"Query: {query}")
print(f"\nRetrieved passages:\n{result}")

Query: What year was the director of Singin in the Rain born?

Retrieved passages:
Donen was born in Columbia, South Carolina, on 13 April 1924. His career in Hollywood was so long, he told Vanity Fair in 2013, “sound was
He was 94. Donen was born April 13, 1924, in South Carolina. To deal with the anti-semitism he said he faced, he retreated
You see, a century ago, Stanley Donen was born in Columbia, South Carolina, the son of a dress shop manager and future movie lover. He'd also be


In [8]:
vague_result = search("things")
print(f"Characters returned: {len(vague_result)}")
print(vague_result[:300])

Characters returned: 2694
Things has also received two MacStories Selects Awards, winning Best App Update of 2018 and Best New Feature of 2023. * **Reminders** integration allows the user to import to-dos from Apple's Reminders app into their Things inbox. * **Add to Things** is an extension on iPhone and iPad that, like Qui


In [9]:
import re
from openai import OpenAI

client = OpenAI()

def run_episode(question: str, max_actions: int = 5) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Question: {question}"}
    ]

    trajectory = []        # stores all steps for logging
    prior_passages = []    # tracks retrieved text for novelty scoring
    action_count = 0

    while action_count < max_actions:
        # --- LLM generates next tokens ---
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            stop=["</search>", "</answer>"],  # stop at action boundaries
            temperature=0.7
        )
        generated = response.choices[0].message.content

        # --- Check for <search> action ---
        if "<search>" in generated:
            query = re.search(r"<search>(.*)", generated, re.DOTALL).group(1).strip()
            passages = search(query)

            # Inject retrieved content as <information> block (paper Section 3.2)
            info_block = f"<information>{passages}</information>"
            messages.append({"role": "assistant", "content": generated + "</search>"})
            messages.append({"role": "user", "content": info_block})

            # --- Compute intermediate rewards (YOUR CONTRIBUTION) ---
            r_relevance = score_relevance(passages, question)
            r_novelty   = score_novelty(passages, prior_passages)
            prior_passages.append(passages)

            trajectory.append({
                "step": action_count,
                "query": query,
                "passages": passages,
                "r_relevance": r_relevance,
                "r_novelty": r_novelty
            })
            action_count += 1

        # --- Check for <answer> action ---
        elif "<answer>" in generated:
            answer = re.search(r"<answer>(.*)", generated, re.DOTALL).group(1).strip()
            return {"answer": answer, "trajectory": trajectory}

        else:
            # Model generated only reasoning — continue
            messages.append({"role": "assistant", "content": generated})
            action_count += 1

    return {"answer": None, "trajectory": trajectory}  # max actions hit

In [10]:
import numpy as np
from openai import OpenAI

client = OpenAI()

def get_embedding(text: str) -> np.ndarray:
    response = client.embeddings.create(
        model="text-embedding-3-small",  # lightweight + cheap
        input=text
    )
    return np.array(response.data[0].embedding)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def score_relevance(passages: str, question: str) -> float:
    q_emb = get_embedding(question)
    p_emb = get_embedding(passages)
    return cosine_similarity(q_emb, p_emb)

def score_novelty(new_passages: str, prior_passages: list) -> float:
    if not prior_passages:
        return 1.0
    new_emb = get_embedding(new_passages)
    prior_embs = [get_embedding(p) for p in prior_passages]
    max_sim = max(cosine_similarity(new_emb, p) for p in prior_embs)
    return 1.0 - max_sim

print("Embedding functions loaded successfully")

Embedding functions loaded successfully


In [11]:
import string

def normalize(text: str) -> str:
    """Lowercase, strip punctuation and extra whitespace."""
    text = text.lower().strip()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())  # collapse whitespace

def score_outcome(predicted: str, ground_truth: str) -> float:
    if predicted is None:
        return 0.0
    pred = normalize(predicted)
    gt   = normalize(ground_truth)
    # Full match first, then substring fallback
    if pred == gt:
        return 1.0
    if gt in pred:   # e.g. agent says "yes they were" but GT is "yes"
        return 0.5
    return 0.0

In [12]:
from datasets import load_dataset

n_questions = 10

dataset = load_dataset("hotpot_qa", "distractor", split="validation")
sample  = dataset.select(range(n_questions))  # start with 200 questions

/Users/alphastar/Northeastern_GitHub/cs5180/research_agent/search-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
import json

logs = []

for item in sample:
    question     = item["question"]
    ground_truth = item["answer"]

    result = run_episode(question)
    r_outcome = score_outcome(result["answer"] or "", ground_truth)

    logs.append({
        "question":     question,
        "ground_truth": ground_truth,
        "predicted":    result["answer"],
        "r_outcome":    r_outcome,
        "trajectory":   result["trajectory"]
    })
    print(f"Q: {question[:60]}... | Outcome: {r_outcome}")

with open("phase1_results.json", "w") as f:
    json.dump(logs, f, indent=2)

Q: Were Scott Derrickson and Ed Wood of the same nationality?... | Outcome: 0.5
Q: What government position was held by the woman who portrayed... | Outcome: 0.0
Q: What science fantasy young adult series, told in first perso... | Outcome: 0.5
Q: Are the Laleli Mosque and Esma Sultan Mansion located in the... | Outcome: 0.5
Q: The director of the romantic comedy "Big Stone Gap" is based... | Outcome: 0.0
Q: 2014 S/S is the debut album of a South Korean boy group that... | Outcome: 1.0
Q: Who was known by his stage name Aladin and helped organizati... | Outcome: 0.5
Q: The arena where the Lewiston Maineiacs played their home gam... | Outcome: 0.0
Q: Who is older, Annie Morton or Terry Richardson?... | Outcome: 0.5
Q: Are Local H and For Against both from the United States?... | Outcome: 0.5


In [14]:
import numpy as np

avg_relevance = [np.mean([s["r_relevance"] for s in log["trajectory"]]) 
                 for log in logs if log["trajectory"]]
avg_novelty   = [np.mean([s["r_novelty"] for s in log["trajectory"]]) 
                 for log in logs if log["trajectory"]]
outcomes      = [log["r_outcome"] for log in logs if log["trajectory"]]

print("Relevance ↔ Outcome correlation:", np.corrcoef(avg_relevance, outcomes)[0,1])
print("Novelty   ↔ Outcome correlation:", np.corrcoef(avg_novelty,   outcomes)[0,1])

Relevance ↔ Outcome correlation: -0.004578629928018818
Novelty   ↔ Outcome correlation: 0.28589426351428193
